In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
rc = ReportClass()


In [2]:
ruta = False

In [ ]:
if ruta: # Si se proporciona una ruta, úsala
    ruta = Path(ruta)
    ruta_kits = ruta / 'data' / 'kits.xlsx'
    df_kits = pd.read_excel(ruta_kits)
else:
    ruta = rc.validar_ruta()
    ruta_kits = ruta / 'data' / 'kits.xlsx'
    df_kits = pd.read_excel(ruta_kits)



# ruta_base =ruta / 'file' / 'BASE_VENTAS.xlsx'
ruta_base = r"C:\Users\Dataa\Desktop\Nueva carpeta\VENTAS_ventas_2024_2025_procesado.xlsx"

# Cargar el archivo de Excel
df_ventas = pd.read_excel(ruta_base)  # Reemplaza con la ruta de tu archivo

# Verifica si falta incluir kit en el archivo kits.xlsx
df_explosion_prueba = pd.merge(df_ventas, df_kits, left_on="PRODUCTO", right_on="KIT", indicator=True, how='left')

productos_con_kit = [
    producto for producto in df_explosion_prueba[df_explosion_prueba['_merge']=='left_only']['PRODUCTO_x'].unique()
    if 'KIT' in str(producto)
]
print(f"Productos con 'KIT' sin correspondencia en df_kits: {productos_con_kit}")


# Unir df_ventas con df_kits para explotar los kits
df_explosion = pd.merge(df_ventas, df_kits, left_on="PRODUCTO", right_on="KIT")

# Agregar una columna para indicar el origen (kit o individual)
df_explosion["ORIGEN"] = "KIT"

# Calcular las cantidades de productos
df_explosion["CANTIDAD_PRODUCTO"] = df_explosion["CANTIDAD"]


# Calcular el valor por producto en los kits
# df_explosion["VALOR_POR_PRODUCTO"] = df_explosion["TOTAL($)"] / df_explosion.groupby("KIT")["PRODUCTO_x"].transform("count")
conteo_facturas = df_explosion.groupby(['PRODUCTO_x','NUMERO_FACTURA'])['NUMERO_FACTURA'].transform('count')
df_explosion["VALOR_POR_PRODUCTO"] = df_explosion['TOTAL($)'] / conteo_facturas


# Filtrar productos individuales
df_ventas_individuales = df_ventas[~df_ventas["PRODUCTO"].str.startswith(("[PCNKIT","[TNGKIT","[B8KIT"))].reset_index(drop=True)
df_ventas_individuales["ORIGEN"] = "INDIVIDUAL"

# Calcular las cantidades de productos
df_ventas_individuales["CANTIDAD_PRODUCTO"] = df_ventas_individuales["CANTIDAD"]
df_ventas_individuales["VALOR_POR_PRODUCTO"] = df_ventas_individuales['TOTAL($)'] 

df_ventas_individuales['PRODUCTO_y'] = df_ventas_individuales['PRODUCTO'] 

df_final_completo = pd.concat([df_explosion, df_ventas_individuales], ignore_index=True)

compras_producto_cliente = (
    df_final_completo[['CLIENTE', 'PRODUCTO_y', 'NUMERO_FACTURA']]
    .drop_duplicates()
    .groupby(['CLIENTE', 'PRODUCTO_y'])
    .agg(Veces_Compra=('NUMERO_FACTURA', 'count'))
    .sort_values(by='Veces_Compra', ascending=False)
).reset_index()



ventas_final = df_final_completo.groupby(['CLIENTE', 'PRODUCTO_y','ORIGEN']).agg(
    PRODUCTO=('PRODUCTO_y', 'first'),
    PRIMERA_COMPRA=('FECHA_FACTURA', 'min'),
    TOTAL_COMPRAS=('NUMERO_FACTURA', 'nunique'),
    ULTIMA_COMPRA=('FECHA_FACTURA', 'max'),
    TELEFONO=('TELEFONO', 'first'),
    TOTAL_CANTIDAD=('CANTIDAD', 'sum')
).reset_index()
ventas_final = ventas_final.merge(compras_producto_cliente, how='left', on=['CLIENTE', 'PRODUCTO_y'])


df_final_completo['PRODUCTO_x'] = df_final_completo['PRODUCTO_x'].fillna(df_final_completo['PRODUCTO_y'])


Productos con 'KIT' sin correspondencia en df_kits: ['[PCN15] KIT HYDRO CLEAN HC']


In [ ]:

# Angelica: Reordenar columnas para que coincidan con el formato solicitado
# df_final_completo = df_final_completo[['NUMERO_FACTURA','FECHA_FACTURA','CLIENTE','CATEGORÍA','PRODUCTO_y','CANTIDAD_PRODUCTO', 'VALOR_POR_PRODUCTO','ORIGEN', 'REFERENCIA','PRODUCTO_x']]

# Daniel: Reordenar columnas para que coincidan con el formato solicitado
df_final_completo = df_final_completo[['NUMERO_FACTURA','FECHA_FACTURA','CLIENTE','CATEGORÍA','PRODUCTO_y','CANTIDAD_PRODUCTO', 'VALOR_POR_PRODUCTO','ORIGEN']]

# df_final_completo.to_csv(r"C:\Users\Dataa\Desktop\dani.csv", index=False, sep=';', decimal=',', encoding='utf-8-sig')

In [11]:

with pd.ExcelWriter(r"C:\Users\Dataa\Desktop\historico_explosionado.xlsx") as file:
    ventas_final.to_excel(file, index=False, sheet_name='Agrupado')
    df_final_completo.to_excel(file, index=False, sheet_name='Base')